# 01 · PEFT 与 LoRA:原理与实战

> 前置:请先完成 [`00_环境准备与GPU检测`](00_环境准备与GPU检测.ipynb) 的环境安装与检测。

本 notebook 分两部分:

1. **原理**:为什么需要参数高效微调(PEFT)、PEFT 家族有哪些、LoRA 到底在做什么。
2. **实战**:用 `peft` 给 `Qwen2.5-1.5B-Instruct` 挂上 LoRA,在我们内置的中文指令数据上真跑几十步,观察 loss 下降和显存占用。

## 一、为什么需要 PEFT?

一个 7B 模型有 70 亿个参数。**全量微调 (full fine-tuning)** 要更新全部参数,代价极高:

- **显存**:除了参数本身(fp16 约 14GB),还要存梯度、优化器状态(Adam 要存两份动量)。经验上全量微调约需 **参数量 × 16 字节** 的显存 —— 7B 就要上百 GB,单张 16GB 卡根本放不下。
- **存储**:每微调一个任务就要存一整份 7B 权重。
- **易过拟合**:小数据上动全部参数容易学偏。

**PEFT (Parameter-Efficient Fine-Tuning)** 的思路:**冻结原模型的绝大部分参数,只训练极少量新增参数**(通常 < 1%),就能把大模型适配到新任务。

常见 PEFT 方法:

| 方法 | 核心思想 |
| --- | --- |
| **LoRA** | 给权重矩阵加一个低秩的「增量」,只训练这个增量 |
| **QLoRA** | 先把底座 4-bit 量化省显存,再在上面做 LoRA(见 02) |
| Prefix / Prompt Tuning | 在输入前拼接一段可训练的「虚拟 token」 |
| Adapter | 在每层之间插入小的可训练模块 |

其中 **LoRA 是目前最主流的方法**,本项目重点讲它。`peft` 库把这些方法统一封装,用法几乎一致。

## 二、LoRA 原理:低秩分解

对某个权重矩阵 $W \in \mathbb{R}^{d \times k}$,微调本质是学一个增量 $\Delta W$,让新权重变成 $W + \Delta W$。

LoRA 的关键假设:**这个增量 $\Delta W$ 的「本征维度」很低**,可以用两个瘦长的小矩阵相乘来近似:

$$\Delta W = B \cdot A, \quad A \in \mathbb{R}^{r \times k},\; B \in \mathbb{R}^{d \times r},\; r \ll \min(d, k)$$

于是前向计算变成:

$$h = W x + \Delta W x = W x + \frac{\alpha}{r}\, B A x$$

- 原权重 $W$ **冻结不动**,只训练 $A$ 和 $B$。
- 参数量从 $d\times k$ 降到 $r\times(d+k)$。比如 $d=k=4096$、$r=8$:从 1600 万降到约 6.5 万,**不到 0.5%**。
- 初始化时 $B=0$,所以训练一开始 $\Delta W=0$,不破坏原模型。

**关键超参数:**

| 超参 | 含义 | 常用取值 |
| --- | --- | --- |
| `r` | 低秩的秩,越大表达力越强、参数越多 | 8 / 16 / 32 |
| `lora_alpha` | 缩放系数,实际缩放为 `alpha/r` | 通常取 `2*r` |
| `target_modules` | 给哪些层加 LoRA(如注意力的 q/k/v/o 投影) | `q_proj`,`v_proj` 等 |
| `lora_dropout` | LoRA 分支上的 dropout,防过拟合 | 0.0 ~ 0.1 |

下面开始动手。

## 三、实战准备:配置与加载模型

首次运行会自动从 HuggingFace 下载 `Qwen2.5-1.5B-Instruct`(约 3GB)。
如果国内下载慢,可在下一格取消注释设置镜像 `HF_ENDPOINT`。

In [ ]:
import os

# 若 HuggingFace 下载慢,取消下面这行注释使用国内镜像
# os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
DATA_PATH = "../data/instructions.jsonl"
OUTPUT_DIR = "../outputs/lora-qwen1.5b"   # 训练好的 LoRA 适配器保存在这里
MAX_LEN = 512

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,   # 5080 支持 bf16
    device_map="cuda",
)
model.config.use_cache = False    # 训练时关闭 KV cache,和梯度检查点兼容

print(f"模型已加载: {MODEL_NAME}")
print(f"参数总量: {model.num_parameters()/1e9:.2f} B")
print(f"当前显存占用: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

## 四、给模型挂上 LoRA

用 `LoraConfig` 描述配置,`get_peft_model` 把 LoRA 适配器注入模型。
注意打印出来的**可训练参数占比**,直观感受 PEFT 的「省」。

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,                     # 低秩的秩
    lora_alpha=16,           # 缩放系数,实际缩放 = alpha/r = 2
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    # 给注意力和 MLP 的投影层加 LoRA(Qwen2 的模块名)
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 五、准备数据

我们的数据是 Alpaca 风格的 `{instruction, input, output}`。要把它转成模型能学的形式:

1. 用 Qwen 的 **chat template** 拼成对话格式(system / user / assistant)。
2. **只对 assistant 的回答计算 loss**(把 prompt 部分的 label 设为 -100),这样模型学的是「怎么回答」,而不是「怎么复述问题」。

> 数据里的回答都以「—— 你的学习小助手」结尾,这个明显的风格特征方便我们在 `03` 里直观检验微调是否生效。

In [ ]:
from datasets import load_dataset

SYSTEM_PROMPT = "你是一个热心的中文学习小助手。"

raw = load_dataset("json", data_files=DATA_PATH, split="train")
print(f"样本数: {len(raw)}")
print("示例:", raw[0])


def build_example(sample):
    """把一条样本转成 input_ids / labels,只对 assistant 回答算 loss。"""
    user_content = sample["instruction"]
    if sample.get("input"):
        user_content += "\n" + sample["input"]

    # 1) 只含 system + user 的前缀(到 assistant 开头),用来定位需要屏蔽的长度
    prefix_ids = tokenizer.apply_chat_template(
        [{"role": "system", "content": SYSTEM_PROMPT},
         {"role": "user", "content": user_content}],
        tokenize=True, add_generation_prompt=True,
    )
    # 2) 完整对话(再接上 assistant 的回答)
    full_ids = tokenizer.apply_chat_template(
        [{"role": "system", "content": SYSTEM_PROMPT},
         {"role": "user", "content": user_content},
         {"role": "assistant", "content": sample["output"]}],
        tokenize=True, add_generation_prompt=False,
    )
    full_ids = full_ids[:MAX_LEN]
    labels = list(full_ids)
    # prompt 部分不计算 loss
    for i in range(min(len(prefix_ids), len(labels))):
        labels[i] = -100
    return {"input_ids": full_ids, "labels": labels,
            "attention_mask": [1] * len(full_ids)}


dataset = raw.map(build_example, remove_columns=raw.column_names)
print("处理后一条:", {k: (v[:8] + ["..."]) for k, v in dataset[0].items()})

## 六、开始训练

用 transformers 的 `Trainer` 训练。这里 `max_steps` 设得很小(方便本机几分钟跑完),
**要真正学出效果,把 `max_steps` 调大到几百步、或改用 `num_train_epochs`。**

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

collator = DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100)

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,   # 等效 batch = 2*4 = 8
    learning_rate=2e-4,              # LoRA 通常用比全量微调大的学习率
    max_steps=60,                    # 演示用;要出效果请调大(如 300+)
    logging_steps=5,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    bf16=True,
    save_strategy="no",              # 演示阶段不中途存盘,最后手动 save
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dataset,
    data_collator=collator,
)

trainer.train()
print(f"训练后显存峰值: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")

## 七、保存 LoRA 适配器

注意保存下来的**只有 LoRA 适配器**(几 MB),不含底座权重。这正是 LoRA 的优势之一:
一个几 GB 的底座可以配无数个几 MB 的适配器。`03` 会加载它做前后对比。

In [ ]:
import os

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# 看看适配器有多小
total = 0
for f in os.listdir(OUTPUT_DIR):
    p = os.path.join(OUTPUT_DIR, f)
    total += os.path.getsize(p)
    print(f"  {f:<32} {os.path.getsize(p)/1024:.1f} KB")
print(f"\n适配器总大小: {total/1024**2:.2f} MB  (对比 1.5B 底座约 3GB)")

## 小结

- **全量微调贵**在要更新+存储全部参数;**PEFT** 只训练极少量新增参数就能适配任务。
- **LoRA** 用低秩分解 $\Delta W = \frac{\alpha}{r} B A$ 表示权重增量,冻结底座、只训练 $A/B$,可训练参数常 < 1%。
- 关键超参:`r`、`lora_alpha`、`target_modules`、`lora_dropout`。
- 保存的适配器只有几 MB。

下一步:[`02_QLoRA原理与实战`](02_QLoRA原理与实战.ipynb) —— 在 LoRA 之上再叠加 4-bit 量化,让 16GB 显存也能微调 7B 模型。